# Property Agents (RN) — Benchmark Completo no Google Colab (GPU)

Este notebook executa **todo o pipeline de benchmark** do projeto diretamente na GPU do Google Colab:

1. **Etapa 1 (`src.run_agents`)**: Executa as 3 arquiteturas sobre o servidor MCP real (`801` imóveis do RN) usando **Ollama na GPU**:
   - `langgraph_unico` (agente ReAct único — baseline)
   - `langgraph_verificador` (gerador + verificador)
   - `langgraph_supervisor` (supervisor + 3 especialistas)
2. **Etapa 2 (`src.metrics`)**: Avalia as respostas com a checagem automática (`checagem_automatica`) + **Juiz Frontier com acesso read-only ao MCP** (`--resume` habilitado).
3. **Etapa 3 (`src.report`)**: Gera o relatório consolidado em PT-BR (`benchmark_summary.md`) e o CSV de auditoria manual (`divergencias_auditoria_manual.csv`), além de empacotar todos os artefatos para download.

### Antes de rodar (`Ambiente de execução -> Executar tudo`)
1. Ative a GPU no Colab: **Ambiente de execução → Alterar o tipo de ambiente de execução → GPU (T4 / L4 / A100)**.
2. Preencha a constante `GEMINI_API_KEY` (ou `JUDGE_PROVIDER` / `JUDGE_API_KEY`) na **célula de configuração logo abaixo**.

In [ ]:
import os
import subprocess
from pathlib import Path

# =============================================================================
# CONSTANTES DE CONFIGURAÇÃO (EDITE AQUI ANTES DE EXECUTAR)
# =============================================================================

# 1. Chave da API do Juiz Frontier (padrão: Gemini 2.5 Flash via Google AI Studio)
GEMINI_API_KEY = "COLOQUE_SUA_CHAVE_GEMINI_AQUI"

# Opções do Juiz Frontier ("gemini" | "openai_compatible" | "ollama")
JUDGE_PROVIDER = "gemini"
JUDGE_MODEL = "gemini-2.5-flash"
JUDGE_BASE_URL = ""  # Ex.: "https://api.groq.com/openai/v1" para Groq
JUDGE_API_KEY = ""  # Preencha se usar openai_compatible (Groq/OpenRouter/OpenAI)
JUDGE_MIN_DELAY_SECONDS = 4.0  # Delay mínimo entre chamadas ao juiz

# 2. Repositório
REPO_URL = "https://github.com/murilodlb/property-agents.git"
REPO_BRANCH = "main"
REPO_DIR = "/content/property-agents"

# 3. Modelo local dos agentes via Ollama na GPU
AGENT_MODEL = "qwen3.5:2b"  # Ex.: "qwen3.5:2b", "qwen3:4b" ou "qwen3:8b"
AGENT_TEMPERATURE = 0.0
AGENT_SEED = 42
AGENT_NUM_CTX = 16384
AGENT_NUM_PREDICT = 1024
AGENT_REASONING = False
AGENT_MAX_CHAMADAS_AGENTE = 8
AGENT_MAX_REVISOES = 1
AGENT_MAX_DELEGACOES = 4
AGENT_TIMEOUT_SECONDS = 600

# 4. Escopo do benchmark e persistência
SISTEMAS = ["langgraph_unico", "langgraph_verificador", "langgraph_supervisor"]
PERGUNTAS_IDS = list(range(1, 21))  # Para smoke test rápido, ex.: [1, 8, 12]
RESUME = True  # Retoma execuções/avaliações parciais sem refazer o que já terminou

# Opcional: salvar resultados direto no Google Drive
SAVE_TO_GOOGLE_DRIVE = False
DRIVE_RESULTS_DIR = "/content/drive/MyDrive/property-agents-results"

# =============================================================================
# Fallback automático para Google Colab Secrets caso GEMINI_API_KEY esteja no placeholder
if not GEMINI_API_KEY or "COLOQUE_SUA_CHAVE" in GEMINI_API_KEY:
    try:
        from google.colab import userdata

        _secret = userdata.get("GEMINI_API_KEY")
        if _secret:
            GEMINI_API_KEY = _secret.strip()
            print("GEMINI_API_KEY carregada do Google Colab Secrets.")
    except Exception:
        pass

if GEMINI_API_KEY and "COLOQUE_SUA_CHAVE" not in GEMINI_API_KEY:
    os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY.strip()
elif JUDGE_PROVIDER == "gemini":
    print("AVISO: preencha GEMINI_API_KEY antes da Etapa 2 (src.metrics).")

os.environ["JUDGE_PROVIDER"] = JUDGE_PROVIDER
os.environ["JUDGE_MODEL"] = JUDGE_MODEL
os.environ["JUDGE_MIN_DELAY_SECONDS"] = str(JUDGE_MIN_DELAY_SECONDS)
if JUDGE_BASE_URL:
    os.environ["JUDGE_BASE_URL"] = JUDGE_BASE_URL
if JUDGE_API_KEY:
    os.environ["JUDGE_API_KEY"] = JUDGE_API_KEY

os.environ["OLLAMA_URL"] = "http://127.0.0.1:11434"
os.environ["AGENT_MODEL"] = AGENT_MODEL
os.environ["AGENT_TEMPERATURE"] = str(AGENT_TEMPERATURE)
os.environ["AGENT_SEED"] = str(AGENT_SEED)
os.environ["AGENT_NUM_CTX"] = str(AGENT_NUM_CTX)
os.environ["AGENT_NUM_PREDICT"] = str(AGENT_NUM_PREDICT)
os.environ["AGENT_REASONING"] = "true" if AGENT_REASONING else "false"
os.environ["AGENT_MAX_CHAMADAS_AGENTE"] = str(AGENT_MAX_CHAMADAS_AGENTE)
os.environ["AGENT_MAX_REVISOES"] = str(AGENT_MAX_REVISOES)
os.environ["AGENT_MAX_DELEGACOES"] = str(AGENT_MAX_DELEGACOES)
os.environ["AGENT_TIMEOUT_SECONDS"] = str(AGENT_TIMEOUT_SECONDS)


def run_streaming(
    cmd: list[str],
    cwd: str | Path | None = None,
    env: dict[str, str] | None = None,
) -> None:
    """Execute a command streaming stdout/stderr live in the notebook output."""
    proc = subprocess.Popen(
        cmd,
        cwd=str(cwd) if cwd is not None else None,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="", flush=True)
    ret = proc.wait()
    if ret != 0:
        raise RuntimeError(f"Comando falhou (exit={ret}): {' '.join(cmd)}")


planned = len(SISTEMAS) * len(PERGUNTAS_IDS)
print(
    f"Configuração pronta | branch={REPO_BRANCH} | modelo={AGENT_MODEL} "
    f"| juiz={JUDGE_PROVIDER}/{JUDGE_MODEL} | execuções planejadas={planned}"
)

## 1. Verificar GPU do Colab e Diretório de Saída

In [ ]:
try:
    run_streaming(["nvidia-smi"])
except Exception as exc:
    print(
        f"AVISO: GPU NVIDIA não detectada via nvidia-smi ({exc}). "
        "Verifique o runtime GPU do Colab."
    )

if SAVE_TO_GOOGLE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    RESULTS_DIR = Path(DRIVE_RESULTS_DIR)
else:
    RESULTS_DIR = Path(REPO_DIR) / "results"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Diretório de resultados: {RESULTS_DIR}")

## 2. Clonar Repositório e Instalar Dependências (`uv` + `nvidia-ml-py`)

In [ ]:
import shutil

repo_path = Path(REPO_DIR)
if not (repo_path / ".git").exists():
    run_streaming(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR])
else:
    run_streaming(["git", "fetch", "origin"], cwd=REPO_DIR)
    run_streaming(["git", "checkout", REPO_BRANCH], cwd=REPO_DIR)
    run_streaming(["git", "reset", "--hard", f"origin/{REPO_BRANCH}"], cwd=REPO_DIR)

os.chdir(REPO_DIR)

for extra_bin in [str(Path.home() / ".local" / "bin"), str(Path.home() / ".cargo" / "bin")]:
    if extra_bin not in os.environ.get("PATH", ""):
        os.environ["PATH"] = f"{extra_bin}:{os.environ.get('PATH', '')}"

if shutil.which("uv") is None:
    run_streaming(["sh", "-c", "curl -LsSf https://astral.sh/uv/install.sh | sh"])

run_streaming(["uv", "sync", "--python", "3.12"], cwd=REPO_DIR)
# Instala nvidia-ml-py (pynvml) no .venv para o ResourceMonitor coletar GPU % e VRAM MB
run_streaming(
    ["uv", "pip", "install", "--python", ".venv/bin/python", "nvidia-ml-py"],
    cwd=REPO_DIR,
)

VENV_PYTHON = str(repo_path / ".venv" / "bin" / "python")
run_streaming(
    [
        VENV_PYTHON,
        "-c",
        "import sys, pynvml; pynvml.nvmlInit(); "
        "print('Python:', sys.version.split()[0], '| GPUs pynvml:', pynvml.nvmlDeviceGetCount())",
    ],
    cwd=REPO_DIR,
)

## 3. Instalar/Iniciar Ollama na GPU e Baixar o Modelo

In [ ]:
import time

import requests

if shutil.which("apt-get") is not None:
    subprocess.run(
        ["sh", "-c", "apt-get update -qq && apt-get install -y -qq pciutils zstd"],
        check=False,
    )

if shutil.which("ollama") is None:
    run_streaming(["sh", "-c", "curl -fsSL https://ollama.com/install.sh | sh"])


def _ollama_alive() -> bool:
    try:
        resp = requests.get("http://127.0.0.1:11434/api/tags", timeout=2)
        return resp.status_code == 200
    except Exception:
        return False


if not _ollama_alive():
    ollama_env = os.environ.copy()
    ollama_env["OLLAMA_HOST"] = "127.0.0.1:11434"
    ollama_env["OLLAMA_FLASH_ATTENTION"] = "1"
    ollama_log = open("/tmp/ollama.log", "w", encoding="utf-8")
    OLLAMA_PROC = subprocess.Popen(
        ["ollama", "serve"],
        env=ollama_env,
        stdout=ollama_log,
        stderr=subprocess.STDOUT,
    )
    for _ in range(30):
        if _ollama_alive():
            break
        time.sleep(1)
    if not _ollama_alive():
        raise RuntimeError("Servidor Ollama não respondeu em /api/tags. Veja /tmp/ollama.log.")
    print("Servidor Ollama iniciado em http://127.0.0.1:11434.")
else:
    print("Servidor Ollama já está ativo.")

run_streaming(["ollama", "pull", AGENT_MODEL])

# Warm-up curto e confirmação de carga em VRAM
requests.post(
    "http://127.0.0.1:11434/api/generate",
    json={"model": AGENT_MODEL, "prompt": "ok", "stream": False, "options": {"num_predict": 4}},
    timeout=120,
).raise_for_status()

ps_data = requests.get("http://127.0.0.1:11434/api/ps", timeout=10).json()
for mod in ps_data.get("models", []):
    size_mb = mod.get("size", 0) / (1024**2)
    vram_mb = mod.get("size_vram", 0) / (1024**2)
    print(f"Modelo ativo: {mod.get('name')} | Total={size_mb:.0f} MB | VRAM={vram_mb:.0f} MB")

## 4. Etapa 1 — Executar os 3 Sistemas de Agentes (`src.run_agents`)

In [ ]:
EXECUTIONS_PATH = RESULTS_DIR / "executions.jsonl"
TRAJECTORIES_PATH = RESULTS_DIR / "trajetorias.jsonl"
AMBIENTE_PATH = RESULTS_DIR / "ambiente.json"

run_agents_cmd = [
    VENV_PYTHON,
    "-m",
    "src.run_agents",
    "--sistemas",
    *SISTEMAS,
    "--perguntas",
    *[str(pid) for pid in PERGUNTAS_IDS],
    "--output",
    str(EXECUTIONS_PATH),
    "--trajetorias",
    str(TRAJECTORIES_PATH),
    "--ambiente",
    str(AMBIENTE_PATH),
    "--model",
    AGENT_MODEL,
]
if RESUME:
    run_agents_cmd.append("--resume")

run_streaming(run_agents_cmd, cwd=REPO_DIR)

n_exec = sum(1 for ln in EXECUTIONS_PATH.read_text(encoding="utf-8").splitlines() if ln.strip())
print(f"Total de registros em {EXECUTIONS_PATH}: {n_exec}")

## 5. Etapa 2 — Avaliação Automática + Juiz Frontier MCP (`src.metrics`)

In [ ]:
if JUDGE_PROVIDER == "gemini":
    gemini_key = os.environ.get("GEMINI_API_KEY", "")
    if not gemini_key or "COLOQUE_SUA_CHAVE" in gemini_key:
        raise ValueError(
            "Preencha GEMINI_API_KEY na célula de constantes no topo do notebook "
            "antes de rodar src.metrics."
        )
elif JUDGE_PROVIDER == "openai_compatible" and not os.environ.get("JUDGE_API_KEY"):
    raise ValueError("Preencha JUDGE_API_KEY para o provedor openai_compatible.")

BENCHMARK_CSV_PATH = RESULTS_DIR / "benchmark_60_execucoes.csv"

metrics_cmd = [
    VENV_PYTHON,
    "-m",
    "src.metrics",
    "--input",
    str(EXECUTIONS_PATH),
    "--output",
    str(BENCHMARK_CSV_PATH),
    "--provider",
    JUDGE_PROVIDER,
    "--delay",
    str(JUDGE_MIN_DELAY_SECONDS),
]
if JUDGE_MODEL:
    metrics_cmd.extend(["--model", JUDGE_MODEL])
if RESUME:
    metrics_cmd.append("--resume")

run_streaming(metrics_cmd, cwd=REPO_DIR)

## 6. Etapa 3 — Gerar Relatório Consolidado e Auditoria Manual (`src.report`)

In [ ]:
from IPython.display import Markdown, display

run_streaming(
    [
        VENV_PYTHON,
        "-m",
        "src.report",
        "--input",
        str(BENCHMARK_CSV_PATH),
        "--out-dir",
        str(RESULTS_DIR),
    ],
    cwd=REPO_DIR,
)

SUMMARY_MD_PATH = RESULTS_DIR / "benchmark_summary.md"
display(Markdown(SUMMARY_MD_PATH.read_text(encoding="utf-8")))

## 7. Inspecionar Dados Gerados e Baixar Artefatos (`.zip`)

In [ ]:
import json
import zipfile

import pandas as pd
from IPython.display import display

if AMBIENTE_PATH.is_file():
    amb = json.loads(AMBIENTE_PATH.read_text(encoding="utf-8"))
    print("Hardware / Ambiente:", json.dumps({
        "cpu": amb.get("cpu"),
        "ram_total_gb": amb.get("ram_total_gb"),
        "gpu": amb.get("gpu"),
        "agente": amb.get("agente"),
    }, ensure_ascii=False, indent=2))

df_bench = pd.read_csv(BENCHMARK_CSV_PATH)
print(f"\nbenchmark_60_execucoes.csv: {len(df_bench)} linha(s)")
display(df_bench[[
    "sistema", "pergunta_id", "categoria", "acerto_automatico", "acerto_juiz",
    "veredito_detalhado", "chamadas_modelo", "num_tool_calls", "latencia_segundos",
    "cpu_pct_medio", "ram_mb_pico", "gpu_pct_medio", "gpu_mem_mb_pico",
]].head(15))

DIV_CSV_PATH = RESULTS_DIR / "divergencias_auditoria_manual.csv"
if DIV_CSV_PATH.is_file():
    df_div = pd.read_csv(DIV_CSV_PATH)
    print(f"\ndivergencias_auditoria_manual.csv: {len(df_div)} linha(s)")
    display(df_div.head(10))

ARTIFACT_FILES = [
    "executions.jsonl",
    "trajetorias.jsonl",
    "ambiente.json",
    "benchmark_60_execucoes.csv",
    "benchmark_summary.md",
    "divergencias_auditoria_manual.csv",
]
ZIP_PATH = RESULTS_DIR / "benchmark_results.zip"
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for name in ARTIFACT_FILES:
        fpath = RESULTS_DIR / name
        if fpath.is_file():
            zf.write(fpath, arcname=name)

print(f"\nPacote gerado: {ZIP_PATH} ({ZIP_PATH.stat().st_size / 1024:.1f} KB)")
try:
    from google.colab import files

    files.download(str(ZIP_PATH))
except ImportError:
    print("Download automático disponível apenas no Google Colab.")